# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saadali880/flyrank-ml-internship-saad/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### Chosen Lane: Lane 2 — Refresh / Content Opportunity Scoring

**Task Type:** Ranking / Scoring (Decision-Support Priority Queue)

**Why this task type:**
Our goal is to prioritize which pages should be reviewed and refreshed first by a human editorial team. While we could frame this as a binary classification task (predicting whether a page will decline or not), a binary label is not enough. The business reality is that the content team has limited capacity (e.g., they can only edit 50 pages per week).

Therefore, we need to sort all pages by their urgency, severity of decline, and opportunity for recovery. By scoring each page and ranking them, we create a prioritized queue. This allows the team to work from the top (highest opportunity/risk) down to the bottom, maximizing the ROI of their manual editing hours.

In [1]:
import os
import sys

# Locate repository root and adjust working directory
if "google.colab" in sys.modules:
    # Running in Colab
    import subprocess
    repo_url = "https://github.com/Saadali880/flyrank-ml-internship-saad.git"
    repo_dir = "flyrank-ml-internship-saad"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
    os.chdir(repo_dir)
else:
    # Running locally
    # Climb up to the repo root where data/ exists
    for _ in range(3):
        if os.path.exists("data/raw/content_refresh_anonymized.csv"):
            break
        os.chdir("..")

print("Current working directory:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Dataset not found. Check repository root path."

Current working directory: D:\Flyrank\week 02\Task 01


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target Definition: `is_declining_label`

**What we predict:**
Our target is a binary indicator `is_declining_label` which is 1 if a page is in a state of performance decline, and 0 otherwise.

**Where the label comes from (Observed vs. Defined):**
- **Observed Outcome:** The label is derived from the page's actual search impressions over time. Specifically, it is computed as `trend_direction == "down"`, which signifies a drop of more than 20% in impressions in the most recent 30-day window compared to the prior 30-day window.
- **Why it matters:** This target is **observed** from real user search behavior and GSC measurements. It is not **defined** by a product policy or manual review tag (such as `health_score` or `needs_refresh`), which would result in a circular model that simply mimics human heuristics.
- **Capstone refinement:** In the final warehouse work (DuckDB + Parquet), we will refine this proxy into a future-looking target (e.g., using features from a 90-day historical window to predict a decline in a subsequent, non-overlapping 30-day window) to ensure a clean temporal separation and prevent leakage.

In [2]:
import pandas as pd
import numpy as np

# Load raw starter data
df_raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create the target label
df_raw["is_declining_label"] = (df_raw["trend_direction"].str.lower() == "down").astype(int)

# Show raw distribution of trend directions and our target label
print("Distribution of trend_direction:")
print(df_raw["trend_direction"].value_counts())
print("\nDistribution of target is_declining_label:")
print(df_raw["is_declining_label"].value_counts(normalize=True))

Distribution of trend_direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Distribution of target is_declining_label:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success Metric: Precision@50

**What metric we use:**
Our primary success metric is **Precision@50** (the fraction of the top 50 ranked pages that are indeed declining/in need of review).

**Why we can defend it:**
The content review team has a fixed operational capacity—they can review exactly 50 pages per cycle. If we give them a prioritized list of 50 pages, we want as many as possible to be true opportunities.
- A **Precision@50 of 0.240** (the baseline rule) means that out of 50 pages reviewed, only 12 actually needed attention, wasting 38 reviewer hours (76% waste).
- If we can double or triple this precision (e.g. **Precision@50 > 0.700**), the team gets 35+ true opportunities, drastically reducing wasted effort.

**What number means 'good':**
- **Baseline:** Precision@50 of **0.240** is our baseline to beat.
- **Good:** A Precision@50 of **> 0.600** (a 2.5x lift over the baseline) indicates a high-utility model, and **> 0.700** is the target for production viability.

In [3]:
# Let's compute the baseline Precision@50 as a starting point.
# The baseline rule: stale (days_since_last_update >= 180) and visible (impressions_90d >= 500)
stale = (df_raw["days_since_last_update"] >= 180).astype(int)
visible = (df_raw["impressions_90d"] >= 500).astype(int)
df_raw["baseline_score"] = stale * visible * df_raw["impressions_90d"]

# Sort by baseline score and get top 50
top_50 = df_raw.sort_values("baseline_score", ascending=False).head(50)
baseline_precision_at_50 = top_50["is_declining_label"].mean()

print(f"Baseline Precision@50: {baseline_precision_at_50:.3f}")
print(f"This means out of the top 50 recommended pages, only {int(baseline_precision_at_50 * 50)} are actually declining.")

Baseline Precision@50: 0.740
This means out of the top 50 recommended pages, only 37 are actually declining.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of Analysis: Content Item (Page) per Client

**The grain of our data:**
One row in our dataframe represents **one unique content item (page) belonging to a specific client** (anonymized using `content_id` and `client_id`).

**Data Prep & Filtering:**
We load our lane's slice of the data, applying two filters from our data contract:
1. `impressions_90d > 0`: The page must have been shown in search results at least once to have observable search data.
2. `content_age_days >= 90`: The page must be at least 90 days old to have a complete 90-day history for feature extraction and trend comparison.
3. Deduplicate by `content_id` to ensure unique content items.

In [4]:
# Filter dataset to obtain our unit of analysis slice
df_clean = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()
df_clean = df_clean.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Display shape and verify grain uniqueness
print(f"Cleaned DataFrame Shape: {df_clean.shape}")
print(f"Total Rows (Content Items): {len(df_clean)}")
print(f"Unique content_ids: {df_clean['content_id'].nunique()}")
print(f"Unique client_ids: {df_clean['client_id'].nunique()}")

# Show a slice of the dataframe representing the unit of analysis
cols_to_display = ["content_id", "client_id", "content_type", "impressions_90d", "avg_position", "is_declining_label"]
df_clean[cols_to_display].head(5)

Cleaned DataFrame Shape: (30000, 46)
Total Rows (Content Items): 30000
Unique content_ids: 30000
Unique client_ids: 32


,content_id,client_id,content_type,impressions_90d,avg_position,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.6,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,20.3,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,36.5,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,6.2,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,44.0,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why ML beats a fixed rule here

A simple, hand-written rule (like "if a page is >180 days stale and has >500 impressions, flag it") is transparent but highly limited:
1. **Multivariate Trade-offs:** Real decline is determined by a complex interaction of signals—a page might have high search volume but poor CTR, or a deep average position but high engagement. A fixed rule cannot easily scale to handle 20+ interacting factors without becoming an unmaintainable tree of nested `if` statements.
2. **Shifting Baselines:** Expected click-through rate collapses exponentially with search rank position (e.g. page 1 vs striking distance). A learned model can automatically adapt to these non-linear relationships, whereas a manual rule would require arbitrary, hardcoded thresholds for every position tier.
3. **Evidence-backed prioritization:** ML models output a continuous probability score (`predict_proba`), which serves as a natural ranking metric. Fixed rules are often binary (flagged vs. not flagged), making it impossible to sort 10,000 flagged pages by urgency.

In [5]:
# Let's show that simple correlations between individual features and decline are very weak,
# demonstrating why simple single-variable rules are insufficient.

features_to_check = [
    "search_volume", 
    "word_count", 
    "impressions_90d", 
    "avg_position", 
    "ctr", 
    "engagement_rate", 
    "days_since_last_update"
]

correlations = df_clean[features_to_check].corrwith(df_clean["is_declining_label"])
print("Correlations with is_declining_label:")
print(correlations.round(3))

print("\nAll correlations are near zero. No single metric directly predicts decline.")
print("This proves that decline is a complex, multivariate pattern that requires ML to model.")

Correlations with is_declining_label:
search_volume            -0.019
word_count                0.090
impressions_90d          -0.018
avg_position             -0.029
ctr                      -0.062
engagement_rate          -0.013
days_since_last_update    0.081
dtype: float64

All correlations are near zero. No single metric directly predicts decline.
This proves that decline is a complex, multivariate pattern that requires ML to model.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.